# 🌍 Introduction to Google Earth Engine in Python

**ESIIL STARS 2026 · Traits and Ecosystems Lab**

In this tutorial you will:

1. Set up a Google Earth Engine account
2. Authenticate and initialize GEE in Python
3. Visualize Landsat satellite imagery of the CSU campus (RGB and false color)
4. Compute and map NDVI (a vegetation index)
5. Explore NLCD land cover and DEM elevation data

**⏱ Estimated time:** 60–90 minutes

**⚠️ Before you start (important):**

1. In the notebook toolbar (top-right), click the current kernel/environment name.
2. Select **`esiil-stars (Python 3.11.15)`**.
3. Do **not** use **`base`**.
4. In the lower-left status bar, click the branch name and create/switch to your own branch (for example, `yourname`) so your work stays separate from `main`.

This way everyone can work through the tutorial independently without conflicts.

---
## Part 0: Setting Up Your Earth Engine Account

Before writing any code, you need a Google Earth Engine account. This is free for research and education. Follow these steps **once** — you won't need to repeat them.

### Step 1: Sign up for Earth Engine

1. Go to **[earthengine.google.com](https://earthengine.google.com/)**
2. Click **"Get Started"** (or **"Sign Up"**) in the top-right corner.
3. Sign in with your **Google account** (your personal Gmail or university Google account both work).
4. You'll be asked to **register a Google Cloud Project**. This is required — Earth Engine runs on Google Cloud infrastructure.

### Step 2: Create a Cloud Project

1. When prompted, click **"Register a Noncommercial or Commercial Cloud project"**.
2. Select **"Unpaid usage"** → **"Academia & Research"**.
3. Click **"Create a new Google Cloud Project"**.
4. Give it a short, memorable name — for example: `esiil-stars-2026` or `ee-yourname`. (Project names must be globally unique, so you may need to try a few.)
5. Accept the Terms of Service and click **Register**.

### Step 3: Note your Project ID

After registering, you'll see a confirmation with your **Project ID** (looks something like `esiil-stars-2026` or `ee-project-abc123`). **Write this down** — you'll need it every time you initialize Earth Engine in Python.

### Common issues

| Problem | Fix |
|---------|-----|
| "You don't have permission" error | Make sure you're signed into the right Google account in your browser. |
| Can't create a Cloud Project | Check that your Google account isn't restricted by an organization. Try a personal Gmail if your university account has restrictions. |
| "Quota exceeded" errors later | Free-tier accounts have a limit on concurrent computations. If this happens, wait a minute and re-run the cell. |
| Registration page looks different than described | Google updates the UI frequently. The core flow is: sign in → create a Cloud Project → register for Earth Engine. Look for "noncommercial" or "academic" options. |

---
## Part 1: Install, Import & Authenticate

Let's get Earth Engine running in this notebook. You only need to authenticate once per Codespace session.

In [ ]:
# Import the libraries we'll use throughout this tutorial
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np

print("✅ All imports successful!")

### Authenticate with Earth Engine

Run the cell below. It will:
1. Print a URL — **click it** (it opens in your browser).
2. Sign in with the same Google account you used to register.
3. Grant access, then **copy the authorization code**.
4. Paste the code back into the box that appears below the cell.

You only do this once per session. If you restart the Codespace, you'll need to re-authenticate.

**Note**: Autication window will pop up at the top!

In [ ]:
# Authenticate — follow the prompts
ee.Authenticate()

### Initialize Earth Engine

Replace `'YOUR-PROJECT-ID'` below with the Cloud Project ID you noted in Part 0 (e.g., `'esiil-stars-2026'`).

If you are not sure what to put for YOUR-PROJECT-ID:
1. Go to https://console.cloud.google.com/,
2. Click the project selector at the top of the page (next to the project name).
3. In the project list, find your Earth Engine project."
4. Copy the value in the Project ID column (not Project Number).
5. Paste that exact Project ID into the code cell below."

In [ ]:
# ⚠️ REPLACE with your actual project ID from Part 0
PROJECT_ID = 'YOUR-PROJECT-ID'

ee.Initialize(project=PROJECT_ID)
print(f"✅ Earth Engine initialized with project: {PROJECT_ID}")

**If you get an error:**
- `"Permission denied"` → Double-check your Project ID. It's case-sensitive.
- `"Not authenticated"` → Re-run the `ee.Authenticate()` cell above.
- `"Earth Engine API has not been enabled"` → Go to [console.cloud.google.com](https://console.cloud.google.com/), select your project, search for "Earth Engine API", and click **Enable**.

---
## Part 2: Define Our Study Area — CSU Campus

We'll use Colorado State University's campus in Fort Collins as a small, familiar area to practice on before scaling up to the full Front Range.

In [ ]:
# CSU campus center coordinates
csu_lon, csu_lat = -105.0844, 40.5734

# Create a 2 km x 2 km box around campus
csu_aoi = ee.Geometry.Point([csu_lon, csu_lat]).buffer(1000).bounds()

# Visualize it on an interactive map
Map = geemap.Map(center=[csu_lat, csu_lon], zoom=14)
Map.addLayer(csu_aoi, {'color': 'blue'}, 'CSU Study Area')
Map

---
## Part 3: Visualizing Landsat Satellite Imagery

Landsat satellites have been imaging the Earth since 1972. We'll use **Landsat 8** (2013–present), which captures light in multiple spectral bands — not just what our eyes see, but also near-infrared and shortwave infrared.

**Key concept:** Landsat Collection 2 Level 2 data has already been corrected for atmospheric effects (surface reflectance). But the pixel values are stored as integers, so we need to apply a **scale factor** to convert them to actual reflectance values (0 to 1).

### Load a summer image from 2016

In [ ]:
def apply_scale_factors(image):
    """Apply Landsat Collection 2 Level 2 scale factors."""
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(optical, overwrite=True)                 .addBands(thermal, overwrite=True)


# Load Landsat 8 for summer 2016
landsat_2016 = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(csu_aoi)
    .filterDate('2016-06-01', '2016-08-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 10))   # Keep only clear scenes
    .map(apply_scale_factors)
    .median()                                    # Composite: median of all clear pixels
    .clip(csu_aoi)
)

print("✅ Landsat 2016 summer composite loaded!")

### True-Color RGB (what your eyes would see)

For Landsat 8, the RGB bands are:
- **B4** = Red
- **B3** = Green
- **B2** = Blue

In [ ]:
# Visualization parameters for true-color RGB
rgb_vis = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0.0,
    'max': 0.3,
}

Map1 = geemap.Map(center=[csu_lat, csu_lon], zoom=14)
Map1.addLayer(landsat_2016, rgb_vis, 'Landsat 2016 — True Color')
Map1.addLayer(csu_aoi, {'color': 'blue'}, 'CSU AOI', shown=False)
Map1

### False-Color Composite (highlights vegetation)

By putting the **near-infrared (B5)** band in the red channel, healthy vegetation appears **bright red** because plants strongly reflect NIR light. This is one of the most useful band combinations in remote sensing.

- **Red channel** ← B5 (Near-Infrared)
- **Green channel** ← B4 (Red)
- **Blue channel** ← B3 (Green)

In [ ]:
# False-color NIR composite
false_color_vis = {
    'bands': ['SR_B5', 'SR_B4', 'SR_B3'],
    'min': 0.0,
    'max': 0.4,
}

Map2 = geemap.Map(center=[csu_lat, csu_lon], zoom=14)
Map2.addLayer(landsat_2016, false_color_vis, 'Landsat 2016 — False Color (NIR)')
Map2

### Now load a second timestamp — summer 2024

In [ ]:
# Load Landsat 8/9 for summer 2024
# (Landsat 9 launched in 2021 — same sensor as Landsat 8)
landsat_2024 = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
    .filterBounds(csu_aoi)
    .filterDate('2024-06-01', '2024-08-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 10))
    .map(apply_scale_factors)
    .median()
    .clip(csu_aoi)
)

# Side-by-side comparison
Map3 = geemap.Map(center=[csu_lat, csu_lon], zoom=14)
Map3.addLayer(landsat_2016, rgb_vis, 'Summer 2016')
Map3.addLayer(landsat_2024, rgb_vis, 'Summer 2024')
Map3

### Save a figure

Let's export the 2024 image as a static figure using `geemap`. This saves a PNG to our `figures/` directory.

In [ ]:
import requests
from PIL import Image
from io import BytesIO

def get_ee_thumbnail(image, vis_params, region, size=512):
    """Fetch a GEE image as a PIL Image for matplotlib.
    This sends a thumbnail request to Google's servers and returns it as an image.
    """
    params = {**vis_params, 'region': region.getInfo(), 'dimensions': size, 'format': 'png'}
    url = image.getThumbURL(params)
    response = requests.get(url)
    return Image.open(BytesIO(response.content))


# Get thumbnails for both timestamps
thumb_2016 = get_ee_thumbnail(landsat_2016, rgb_vis, csu_aoi)
thumb_2024 = get_ee_thumbnail(landsat_2024, rgb_vis, csu_aoi)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(thumb_2016)
axes[0].set_title('CSU Campus — Summer 2016', fontsize=13)
axes[0].axis('off')

axes[1].imshow(thumb_2024)
axes[1].set_title('CSU Campus — Summer 2024', fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('figures/csu_landsat_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure saved to figures/csu_landsat_comparison.png")

### 🏋️ Exercise 1: Your Turn!

Try the following on your own. Add code cells below for each task.

**1a.** Load a **winter** Landsat composite for CSU campus (use dates `'2024-01-01'` to `'2024-03-31'`). How does it look different from the summer image?

**1b.** Display the winter image in **false color (NIR)**. Can you still see vegetation? What does snow look like in false color?

*Hint: Copy the code cells above and just change the dates and variable names.*

In [ ]:
# ✏️ Exercise 1a: Load winter 2024 Landsat
# Your code here...



In [ ]:
# ✏️ Exercise 1b: Display winter image in false color
# Your code here...



---
## Part 4: Computing NDVI — A Vegetation Index

**NDVI** (Normalized Difference Vegetation Index) is the most widely used remote sensing index for measuring vegetation health. The formula is:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

- **NDVI ≈ 0.0–0.1:** Bare soil, water, pavement
- **NDVI ≈ 0.2–0.4:** Sparse vegetation, grass
- **NDVI ≈ 0.5–0.9:** Dense, healthy vegetation

The key insight: healthy green plants absorb red light (for photosynthesis) and strongly reflect near-infrared light. Stressed or dead vegetation reflects less NIR.

In [ ]:
def compute_ndvi(image):
    """Compute NDVI from Landsat 8/9 bands."""
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return image.addBands(ndvi)


# Compute NDVI for both timestamps
ndvi_2016 = compute_ndvi(landsat_2016)
ndvi_2024 = compute_ndvi(landsat_2024)

# Visualization parameters for NDVI
ndvi_vis = {
    'bands': ['NDVI'],
    'min': -0.1,
    'max': 0.8,
    'palette': ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']
}

# Map it
Map4 = geemap.Map(center=[csu_lat, csu_lon], zoom=14)
Map4.addLayer(ndvi_2016, ndvi_vis, 'NDVI — Summer 2016')
Map4.addLayer(ndvi_2024, ndvi_vis, 'NDVI — Summer 2024')
Map4.add_colorbar(ndvi_vis, label='NDVI')
Map4

### Save the NDVI comparison figure

In [ ]:
# Reuse our get_ee_thumbnail function from earlier
ndvi_thumb_2016 = get_ee_thumbnail(ndvi_2016, ndvi_vis, csu_aoi)
ndvi_thumb_2024 = get_ee_thumbnail(ndvi_2024, ndvi_vis, csu_aoi)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(ndvi_thumb_2016)
axes[0].set_title('NDVI — Summer 2016', fontsize=13)
axes[0].axis('off')

axes[1].imshow(ndvi_thumb_2024)
axes[1].set_title('NDVI — Summer 2024', fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('figures/csu_ndvi_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure saved to figures/csu_ndvi_comparison.png")

### 🏋️ Exercise 2: Your Turn!

**2a.** Compute the **NDVI difference** between 2024 and 2016. Where has vegetation increased? Where has it decreased?

*Hint:*
```python
ndvi_change = ndvi_2024.select('NDVI').subtract(ndvi_2016.select('NDVI')).rename('NDVI_change')
```

Use a **diverging** color palette (red = loss, blue/green = gain):
```python
change_vis = {
    'bands': ['NDVI_change'],
    'min': -0.3,
    'max': 0.3,
    'palette': ['#d73027', '#f4f4f4', '#1a9850']
}
```

**2b.** What areas show the biggest changes? Can you guess why? (Think about new buildings, new landscaping, etc.)

In [ ]:
# ✏️ Exercise 2a: Compute and map NDVI change
# Your code here...



---
## Part 5: Land Cover with NLCD

The **National Land Cover Database (NLCD)** classifies every 30m pixel in the US into land cover types (forest, developed, agriculture, etc.). We'll use this dataset extensively in our Front Range project, so let's get familiar with it here.

NLCD is available in GEE as a multi-year image collection. Each image has a `landcover` band with integer class codes.

In [ ]:
# Load NLCD for two years
nlcd_2001 = ee.Image('USGS/NLCD_RELEASES/2021_REL/NLCD/2001').select('landcover').clip(csu_aoi)
nlcd_2021 = ee.Image('USGS/NLCD_RELEASES/2021_REL/NLCD/2021').select('landcover').clip(csu_aoi)

# NLCD has its own built-in color palette — GEE knows how to display it
Map5 = geemap.Map(center=[csu_lat, csu_lon], zoom=14)
Map5.addLayer(nlcd_2001, {}, 'NLCD 2001')
Map5.addLayer(nlcd_2021, {}, 'NLCD 2021')
Map5.add_legend(title='NLCD Land Cover', builtin_legend='NLCD')
Map5

### Understanding NLCD classes

Here are the classes you'll see most around CSU:

| Code | Class | Color |
|------|-------|-------|
| 11 | Open Water | Dark Blue |
| 21 | Developed, Open Space | Light Pink |
| 22 | Developed, Low Intensity | Pink |
| 23 | Developed, Medium Intensity | Red |
| 24 | Developed, High Intensity | Dark Red |
| 41 | Deciduous Forest | Light Green |
| 42 | Evergreen Forest | Dark Green |
| 52 | Shrub/Scrub | Tan |
| 71 | Grassland/Herbaceous | Yellow-Green |
| 81 | Pasture/Hay | Yellow |
| 82 | Cultivated Crops | Brown |

Toggle between the 2001 and 2021 layers on the map. Can you see how development has expanded around campus?

### Count pixels by land cover class

This is a preview of the kind of analysis we'll do at the Front Range scale — counting how many pixels of each class fall within our area.

In [ ]:
# Count pixels per NLCD class in 2001 vs 2021
def count_classes(nlcd_image, year):
    """Count pixels per NLCD class and return as a dictionary."""
    histogram = nlcd_image.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=csu_aoi,
        scale=30,
        maxPixels=1e9
    ).getInfo()

    counts = histogram['landcover']
    print(f"\n--- NLCD {year} ---")
    for class_code, count in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"  Class {class_code}: {count} pixels")
    return counts


counts_2001 = count_classes(nlcd_2001, 2001)
counts_2021 = count_classes(nlcd_2021, 2021)

In [ ]:
# Quick bar chart comparison
import pandas as pd

# Build a DataFrame from the counts
nlcd_names = {
    '11': 'Water', '21': 'Dev-Open', '22': 'Dev-Low',
    '23': 'Dev-Med', '24': 'Dev-High', '41': 'Deciduous',
    '42': 'Evergreen', '52': 'Shrub', '71': 'Grassland',
    '81': 'Pasture', '82': 'Crops'
}

all_classes = sorted(set(list(counts_2001.keys()) + list(counts_2021.keys())))
df = pd.DataFrame({
    'Class': [nlcd_names.get(c, f'Other ({c})') for c in all_classes],
    '2001': [counts_2001.get(c, 0) for c in all_classes],
    '2021': [counts_2021.get(c, 0) for c in all_classes],
})

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(df))
width = 0.35
ax.bar([i - width/2 for i in x], df['2001'], width, label='2001', color='#66c2a5')
ax.bar([i + width/2 for i in x], df['2021'], width, label='2021', color='#fc8d62')
ax.set_xticks(x)
ax.set_xticklabels(df['Class'], rotation=45, ha='right')
ax.set_ylabel('Pixel Count')
ax.set_title('NLCD Land Cover Change — CSU Campus Area (2001 vs 2021)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('figures/csu_nlcd_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure saved to figures/csu_nlcd_comparison.png")

### 🏋️ Exercise 3: Your Turn!

**3a.** Expand the study area to a **5 km buffer** around CSU and re-run the NLCD comparison. Do you see more agricultural land at this scale?

*Hint:*
```python
csu_wide = ee.Geometry.Point([csu_lon, csu_lat]).buffer(5000).bounds()
```
Remember to clip the NLCD images to this new geometry.

**3b.** Which land cover class **increased** the most between 2001 and 2021? Which **decreased** the most? Write your answer as a comment in the code cell.

In [ ]:
# ✏️ Exercise 3a: Wider AOI NLCD comparison
# Your code here...



---
## Part 6: Elevation with the USGS 3DEP DEM

The **3D Elevation Program (3DEP)** provides a 10-meter resolution Digital Elevation Model (DEM) for the entire US. Elevation is critical for our Front Range project because we're studying how forest cover changes along an elevation gradient.

Let's look at the terrain around CSU campus — even though Fort Collins is relatively flat, you'll see the foothills rising to the west.

In [ ]:
# Load the 10m DEM — use a wider area to see the foothills
foothills_aoi = ee.Geometry.Point([csu_lon, csu_lat]).buffer(15000).bounds()

dem = ee.Image('USGS/3DEP/10m').clip(foothills_aoi)

# Elevation visualization
elev_vis = {
    'min': 1500,
    'max': 2200,
    'palette': ['#2b83ba', '#abdda4', '#ffffbf', '#fdae61', '#d7191c']
}

Map6 = geemap.Map(center=[csu_lat, csu_lon - 0.05], zoom=12)
Map6.addLayer(dem, elev_vis, 'Elevation (m)')
Map6.add_colorbar(elev_vis, label='Elevation (m)')
Map6

### Compute slope from the DEM

Slope tells us how steep the terrain is — useful for understanding where forests can grow.

In [ ]:
# Compute slope in degrees
slope = ee.Terrain.slope(dem)

slope_vis = {
    'min': 0,
    'max': 30,
    'palette': ['#f7fcf5', '#74c476', '#238b45', '#00441b']
}

Map7 = geemap.Map(center=[csu_lat, csu_lon - 0.05], zoom=12)
Map7.addLayer(slope, slope_vis, 'Slope (degrees)')
Map7.add_colorbar(slope_vis, label='Slope (degrees)')
Map7

### Combine DEM + NLCD: A preview of the Front Range analysis

This is the core idea behind our project — looking at how land cover varies with elevation.

In [ ]:
# Sample elevation values at every NLCD pixel in our wider area
nlcd_foothills = ee.Image('USGS/NLCD_RELEASES/2021_REL/NLCD/2021') \
    .select('landcover').clip(foothills_aoi)

# Combine DEM and NLCD into one image
combined = dem.addBands(nlcd_foothills)

# Sample 2000 random points
samples = combined.sample(
    region=foothills_aoi,
    scale=30,
    numPixels=2000,
    seed=42,
    geometries=False
)

# Convert to pandas
df_samples = geemap.ee_to_df(samples)
print(f"Sampled {len(df_samples)} points")
df_samples.head()

In [ ]:
# Plot: How does land cover change with elevation?
import seaborn as sns

# Map NLCD codes to names
nlcd_name_map = {
    11: 'Water', 21: 'Dev-Open', 22: 'Dev-Low', 23: 'Dev-Med',
    24: 'Dev-High', 41: 'Deciduous', 42: 'Evergreen', 43: 'Mixed Forest',
    52: 'Shrub', 71: 'Grassland', 81: 'Pasture', 82: 'Crops', 90: 'Wetland'
}
nlcd_color_map = {
    'Dev-Open': '#eda1a1', 'Dev-Low': '#e87f7f', 'Dev-Med': '#ed0000',
    'Dev-High': '#ab0000', 'Evergreen': '#1c6330', 'Deciduous': '#68ab63',
    'Grassland': '#e3d28e', 'Shrub': '#ccb879', 'Pasture': '#dbd93d',
    'Crops': '#ab7028', 'Water': '#476ba1', 'Wetland': '#bad9eb'
}

df_samples['Land Cover'] = df_samples['landcover'].map(nlcd_name_map)
df_samples = df_samples.dropna(subset=['Land Cover'])

# Keep only the most common classes for readability
top_classes = df_samples['Land Cover'].value_counts().head(6).index.tolist()
df_plot = df_samples[df_samples['Land Cover'].isin(top_classes)]

fig, ax = plt.subplots(figsize=(10, 5))
colors = [nlcd_color_map.get(c, '#999999') for c in top_classes]
sns.boxplot(data=df_plot, x='Land Cover', y='elevation', palette=colors, ax=ax)
ax.set_ylabel('Elevation (m)')
ax.set_xlabel('')
ax.set_title('Land Cover vs Elevation — CSU to Foothills')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('figures/csu_landcover_vs_elevation.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure saved to figures/csu_landcover_vs_elevation.png")

### 🏋️ Exercise 4: Your Turn!

**4a.** Create an **elevation histogram** using `plt.hist()` for the sampled points. What's the elevation range in our 15 km study area?

*Hint:*
```python
plt.hist(df_samples['elevation'], bins=30, color='steelblue', edgecolor='white')
plt.xlabel('Elevation (m)')
```

**4b.** In the boxplot above, which land cover class sits at the **highest** elevation? Does that make ecological sense?

In [ ]:
# ✏️ Exercise 4a: Elevation histogram
# Your code here...



---
## ✅ Wrap-Up: Save Your Work!

You've learned how to:
- Authenticate and initialize Google Earth Engine
- Load and visualize Landsat satellite imagery (RGB and false color)
- Compute vegetation indices (NDVI)
- Work with NLCD land cover data
- Work with DEM elevation data
- Combine datasets for cross-variable analysis

These are the exact same tools and techniques we'll use for the Front Range project — just at a larger scale.

### Save your work to your branch

Open the terminal (`` Ctrl+` ``) and run:

```bash
git add .
git commit -m "Complete GEE tutorial with exercises"
git push -u origin yourname-gee-tutorial
```

**⚠️ Do NOT push to `main`.** Your branch keeps your tutorial work separate so that every team member can complete it independently. Once everyone has finished, we'll move on to the actual project notebooks.

### Next steps
→ Start working through the analysis notebooks in `notebooks/`!